In [ ]:
import pandas as pd
import numpy as np
import joblib

features = pd.read_csv("../data/customer_features.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

test_df = test.merge(features, on="cust_id", how="left")
print(test_df.isna().any(axis=1).sum())
print(test_df.isna().sum()[test_df.isna().sum() > 0])
test_df = test_df.fillna(0)

# load feature columns
feature_cols = joblib.load("../models/feature_columns.pkl")
X_test = test_df[feature_cols]

# load Churn classifiers 
churn_lgb = joblib.load("../models/churn_lgb_model.pkl")
churn_xgb = joblib.load("../models/churn_xgb_model.pkl")
churn_cat = joblib.load("../models/churn_cat_model.pkl")
iso_lgb   = joblib.load("../models/iso_lgb.pkl")
iso_xgb   = joblib.load("../models/iso_xgb.pkl")
iso_cat   = joblib.load("../models/iso_cat.pkl")

# generating churn classifier ensemble
def predict_churn_proba(X):
    p_lgb = iso_lgb.transform(churn_lgb.predict_proba(X)[:, 1])
    p_xgb = iso_xgb.transform(churn_xgb.predict_proba(X)[:, 1])
    p_cat = iso_cat.transform(churn_cat.predict_proba(X)[:, 1])
    return (p_lgb + p_xgb + p_cat) / 3

p_return = predict_churn_proba(X_test)

# load revenue 2-stage model
revenue_model = joblib.load("../models/rev_model_2stage.pkl")
threshold = joblib.load("../models/best_threshold.pkl")

log_rev_pred = revenue_model.predict(X_test)
rev_pred     = np.maximum(log_rev_pred ** 2, 0)

two_stage_pred = np.where(
    p_return < threshold,
    0,
    p_return * rev_pred
)

# Pure regressor blend - if other model better than LGB => TO BE SET MANUALLY
pure_lgb  = joblib.load("../models/pure_lgb.pkl")
best_beta = joblib.load("../models/best_beta_lgb.pkl")

pure_preds_test = np.maximum(pure_lgb.predict(X_test) ** 2, 0)

final_pred = best_beta * pure_preds_test + (1 - best_beta) * two_stage_pred

print(f"β = {best_beta:.2f}")
print(f"Average predictions: {final_pred.mean():.2f}")
print(f"Zero predictions: {(final_pred == 0).sum()}")

# Submission
import os
os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    "cust_id":    test_df["cust_id"],
    "prediction": final_pred
})

submission.to_csv("../submissions/submission.csv", index=False)

print(submission.shape)
print(submission.head())
print(f"Threshold = {threshold:.4f}")
print(submission["prediction"].describe())
print(f"Zero predictions: {(submission['prediction'] == 0).sum()}")